1. Get all products priced between 100 and 500, category either 'Electronics' or 'Home'.

In [0]:
df = df.filter((col('price') > 100) & (col('price') < 500) & (col('category').isin('Electronics', 'Home')))

2. For each department, show the average salary alongside every individual employee's row (not collapsed).

In [0]:
df = df.withColumn('avg_salary', avg('salary').over(Window.partitionBy('department')))

3. Find users who logged in on more than 3 distinct days this month.

In [0]:
result = df.groupBy('user_id').agg(countDistinct('login_date').alias('distinct_days')).filter(col('distinct_days') > 3)

4. Show the difference between each employee's salary and their department's average salary, per employee.

In [0]:
df = df.withColumn('avg_salary', avg('salary').over(Window.partitionBy('department')))
df = df.withColumn('salary_diff', col('salary') - col('avg_salary'))
df = df.select('emp_id', 'salary_diff')

5. Join products and inventory tables — find inventory records where the product ID does NOT exist in products (orphaned records).

In [0]:
df_new = inventory.join(products, on='product_id', how='left').filter(col('products.product_id').isNull())

6. Join two DataFrames with no shared column — explain the different approaches: cross join, joining on differently-named keys, row-position join, and union (when "combine" actually means stack, not join).

In [0]:
# Cross join (every combination)
result = df1.crossJoin(df2)

# Differently-named keys
result = df1.join(df2, df1.customer_id == df2.cust_id, how='inner')

# Row-position join (fragile, last resort)
from pyspark.sql.functions import monotonically_increasing_id
df1 = df1.withColumn('row_id', monotonically_increasing_id())
df2 = df2.withColumn('row_id', monotonically_increasing_id())
result = df1.join(df2, on='row_id', how='inner').drop('row_id')

# Union (stacking rows, same schema)
result = df1.union(df2)